In [1]:
import numpy as np

import pandas as pd
pd.options.plotting.backend = "plotly"

import os
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler
import scipy
import matplotlib.pyplot as plt
import seaborn as sns


import warnings
warnings.filterwarnings('once')

In [2]:
import tensorflow as tf
tf.__version__

'2.17.0'

# Processamento e seleção da melhor coluna

In [27]:
best_column = 'NDCG@15'
metric_type = ["Prec", "Rec", "F1_Score", "Hit_Rate", "NDCG"]
top_k = [3, 5, 10, 20]

main_path = "results/metrics/validation"
main_file = os.listdir(main_path)

dataframes = []

for dataset_name in main_file:
        
    recommender_files = os.listdir(os.path.join(main_path, dataset_name))
    for recommender_name in recommender_files:
                
        metrics_path = os.path.join(main_path, dataset_name, recommender_name, "metrics.csv")
        metrics_aux = pd.read_csv(metrics_path, sep=';')
        
        metrics_aux.insert(0,'Recommender','')
        metrics_aux['Recommender'] = recommender_name

        metrics_aux.insert(0,'Dataset','')
        metrics_aux['Dataset'] = dataset_name
        
        dataframes.append(metrics_aux)
                
metrics_df = pd.concat(dataframes, ignore_index=True)

if (best_column == 'Mean'):
    metrics_df.insert(3,'Mean', metrics_df.mean(axis=1))
    
if best_column in metrics_df.columns:
    saida = metrics_df.loc[metrics_df.reset_index().groupby(['Dataset', 'Recommender'])[best_column].idxmax()].reset_index(drop=True)
else:
    raise Exception("Coluna '" + best_column + "' não encontrada")

# Análise de resultados

## 1- Geral

In [28]:
pd.options.display.max_colwidth = 200
df_display = saida[saida.Recommender != "ALS"][["Dataset", "Recommender", "Parameters", "NDCG@20"]]
df_display = df_display[saida.Recommender != "BPR"]
display(df_display)

C:\Users\Rafael Tofoli\AppData\Local\Temp\ipykernel_19620\140571426.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_display = df_display[saida.Recommender != "BPR"]


,Dataset,Recommender,Parameters,NDCG@20
2,BestBuy,Item2Vec_itemSim,learning_rate=0.25@lr_decay=0.5@min_learning_rate=0.025@negative_exp=-0.75@subsample=0.001@w_size=-1@epochs=120,0.036543
5,CiaoDVD,Item2Vec_itemSim,learning_rate=0.025@lr_decay=0.3@min_learning_rate=0.025@negative_exp=-0.75@subsample=0.001@w_size=-1@epochs=80,0.015102
6,CiaoDVD,TimeI2V_Disc,learning_rate=0.25@lr_decay=0.5@min_learning_rate=0.025@min_time_diff=86500@negative_exp=0.75@subsample=0.001@time_exp=1.5@w_size=-1@epochs=100,0.017245
7,CiaoDVD,TimeI2V_Disc_Aug,learning_rate=0.025@lr_decay=0.5@min_learning_rate=0.025@min_time_diff=300@negative_exp=0.75@subsample=0.001@time_exp=1.5@w_size=-1@epochs=60,0.016437
10,MovieLens,Item2Vec_itemSim,learning_rate=0.25@lr_decay=0.2@min_learning_rate=0.0025@negative_exp=-0.75@subsample=0.001@w_size=20@epochs=40,0.036920
11,MovieLens,TimeI2V_Cont,curve_exp=-1@learning_rate=0.25@lr_decay=0.2@min_learning_rate=0.025@min_time_diff=86500@min_weight=0.1@negative_exp=-0.75@subsample=0.0001@w_size=20@weight_floor=0.3@epochs=80,0.053846
12,MovieLens,TimeI2V_Disc,learning_rate=0.25@lr_decay=0.2@min_learning_rate=0.025@min_time_diff=300@negative_exp=-0.75@subsample=0.001@time_exp=1.5@w_size=20@epochs=100,0.036336
13,MovieLens,TimeI2V_Disc_Aug,learning_rate=0.25@lr_decay=0.2@min_learning_rate=0.025@min_time_diff=86500@negative_exp=-0.75@subsample=0.001@time_exp=1.5@w_size=20@epochs=20,0.044329
16,RetailRocket-Transactions,Item2Vec_itemSim,learning_rate=0.25@lr_decay=0.5@min_learning_rate=0.025@negative_exp=1@subsample=0.01@w_size=-1@epochs=120,0.050296
17,RetailRocket-Transactions,TimeI2V_Disc,learning_rate=0.025@lr_decay=0.2@min_learning_rate=0.0025@min_time_diff=86500@negative_exp=0.75@subsample=0.01@time_exp=1.5@w_size=20@epochs=10,0.056007


## 2 - Por parâmetros

In [35]:
pd.set_option('display.max_rows', 100)

def metrics_per_parameter(df, df_name, parameter_name):

    def extract_parameter_value(param_str, target_param):
        params = dict(item.split('=') for item in param_str.split('@'))
        return params.get(target_param, None)

    if df_name != 'all':
        df = df[df['Dataset'] == df_name].copy()
        
    df[parameter_name] = df['Parameters'].apply(lambda x: extract_parameter_value(x, parameter_name))
    
    best_values = (
        df.groupby(['Dataset', 'Recommender', parameter_name])['NDCG@20']
        .max()
        .reset_index()
        .sort_values(by=['Dataset', 'Recommender', 'NDCG@20'], ascending=[True, True, False])
    )
    
    return best_values

metrics_per_parameter(metrics_df, 'all', 'time_exp')

,Dataset,Recommender,time_exp,NDCG@20
1,CiaoDVD,TimeI2V_Disc,1.5,0.017245
0,CiaoDVD,TimeI2V_Disc,0,0.015169
3,CiaoDVD,TimeI2V_Disc_Aug,1.5,0.016682
2,CiaoDVD,TimeI2V_Disc_Aug,0,0.016081
5,MovieLens,TimeI2V_Disc,1.5,0.036336
4,MovieLens,TimeI2V_Disc,0,0.033933
7,MovieLens,TimeI2V_Disc_Aug,1.5,0.044343
6,MovieLens,TimeI2V_Disc_Aug,0,0.042370
9,RetailRocket-Transactions,TimeI2V_Disc,1.5,0.058917
8,RetailRocket-Transactions,TimeI2V_Disc,0,0.054023


# Geração de gráficos

In [ ]:
def function_1(x):
    if(len(x.split('@')) > 1): 
        x = x.split('@')[1]
    return x

curr_metric = "Hit_Rate"

main_path = "results/metrics"
main_file = os.listdir(main_path)

for dataset_name in main_file:
    
    saida_aux = saida[saida['Dataset'] == dataset_name] 
        
    my_regex = "Recommender|" +  curr_metric + ".*"
    df_aux = saida_aux.filter(regex=(my_regex))
    
    df_aux = df_aux.set_index('Recommender')
    df_aux = df_aux.rename(columns=function_1)
    
    marker_symbols = ['circle', 'square', 'diamond', 'cross', 'triangle-up', 'triangle-down', 'star', 'hexagram']

    fig = go.Figure()

    for i, recomendador in enumerate(df_aux.index):
        fig.add_trace(
            go.Scatter(
                x=df_aux.columns,  # Colunas de métricas
                y=df_aux.loc[recomendador],  # Pontuações do recomendador
                mode='lines+markers',
                marker=dict(symbol=marker_symbols[i], size=7, line=dict(color='black', width=0.5)),
                name=recomendador
            )
        )
    
    fig.update_layout(
    title=dataset_name+" - "+curr_metric,
    title_x=0.15,
    title_y=0.8,
    showlegend=True,
    height=400,
    width=600,
    yaxis_title="",
    xaxis_title="",
    legend_title="Recommenders",
    font=dict(
        family="Helvetica",
        size=12,
        color="black"
        )
    )

    fig.show()

In [ ]:
'''def function_1(x):
    if len(x.split('@')) > 1: 
        x = x.split('@')[1]
    return x

curr_metric = "NDCG"

main_path = "results/metrics"
main_file = os.listdir(main_path)

# Determine the grid size
rows = 1
cols = 1
num_plots = len(main_file)

# Define colors for each recommender
color_map = {
    'ALS - UI': '#1f77b4',
    'ALS - II': '#ff7f0e',
    'ALS - WS': '#2ca02c',
    'BPR - UI': '#d62728',
    'BPR - II': '#9467bd',
    'BPR - WS': '#bcbd22',
}

# Create subplots with adjusted width and height
fig = make_subplots(
    rows=rows, 
    cols=cols, 
    subplot_titles=main_file,
    vertical_spacing=0.05,  # Decrease vertical space between plots
    horizontal_spacing=0.05  # Decrease horizontal space between plots
)

# Iterate over datasets and create subplots
for idx, dataset_name in enumerate(main_file):
    row = (idx // cols) + 1
    col = (idx % cols) + 1
    
    saida_aux = saida[saida['Dataset'] == dataset_name] 
    my_regex = "Recommender|" +  curr_metric + ".*"
    df_aux = saida_aux.filter(regex=(my_regex))
    
    df_aux["Recommender"] = df_aux["Recommender"].replace(["ALS", "ALS_itemSim", "ALS_weighted", "BPR", "BPR_itemSim", "BPR_weighted"], 
                                                  ["ALS - UI", "ALS - II", "ALS - WS", "BPR - UI", "BPR - II", "BPR - WS"])
        
    df_aux = df_aux.set_index('Recommender')
    df_aux = df_aux.rename(columns=function_1)
    
    marker_symbols = ['circle', 'square', 'diamond', 'cross', 'triangle-up', 'triangle-down', 'star', 'hexagram']

    for i, recomendador in enumerate(df_aux.index):
        fig.add_trace(
            go.Scatter(
                x=df_aux.columns,  # Colunas de métricas
                y=df_aux.loc[recomendador],  # Pontuações do recomendador
                mode='lines+markers',
                marker=dict(symbol=marker_symbols[i], size=7, color=color_map.get(recomendador, 'black'), line=dict(color='black', width=0.5)),
                name=recomendador,
                showlegend=(idx == 6)  # Show legend only for the specified subplot
            ),
            row=row,
            col=col
        )

fig.update_layout(
    height=1400,  # Increase height
    width=800,   # Decrease width
    title_text="",  # Global title
    showlegend=True,
    font=dict(
        family="Helvetica",
        size=12,
        color="black"
    ),
    legend_title="Recommenders",
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.04,  
        xanchor='center',
        x=0.5,
        font=dict(
            size=12.5
        )
    )
)

fig.show()'''

# Geração de tabelas - Divisão por Top-N

In [ ]:
# Gera uma tabela para todos os datasets em todos os Top-N diferentes
# Retorna o melhor parametro de cada dataset

def function_1(x):
   x = x.split('@')[0]
   return x

names_dict = {"Prec": "Precision", "Rec": "Recall", "F1_Score":"F1-Score", "Hit_Rate": "Hit-Rate"}

results_path = "latex_results"

string_final = ""

if not os.path.exists(results_path):
    os.makedirs(results_path)
    
main_file = os.listdir(main_path)

for dataset_name in main_file:
    
    saida_aux = saida[saida['Dataset'] == dataset_name] 
      
    string_final = string_final + ("=========== DATASET: " + dataset_name + " ===========\n\n\n")
    string_final = string_final + ("=========== MELHORES PARÂMETROS ===========\n\n")
    string_final = string_final + (saida[["Recommender", "Parameters"]].to_string())
    
    for curr_k in top_k:
        
        aux = '\n\n------ Top' + str(curr_k) + ' ------\n\n'
        string_final = string_final + (aux)

        #Pre processamento das colunas
        my_regex = "Recommender|" +  str(curr_k) + "$"
        df_aux = saida_aux.filter(regex=(my_regex))
        df_aux = df_aux.round(4)
        df_aux = df_aux.rename(columns=function_1)
        df_aux = df_aux.rename(columns=names_dict)

        # Tranforma a tabela em Latex
        capt = "Top " + str(curr_k) + " recommendation"
        string_final = string_final + (df_aux.style.hide(axis="index").set_caption(capt).to_latex())
        
        
    string_final = string_final + ("\n\n\n\n")

f = open(os.path.join(results_path, "saida.txt"), "w") 
f.write(string_final)
f.close()

# Geração de tabelas - Tabela unificada

In [ ]:
def pre_proc(df, comp_metric):
    
    #Seleciona apenas a metrica relevante
    df_aux = df.loc[:, ["Dataset", "Recommender", comp_metric]].copy()

    #Separa a coluna de recomendadores em dois
    df_aux[['Embedding', 'Recommender']]= df_aux['Recommender'].str.split('_', expand=True)

    #Preenche os valores nulos
    df_aux['Recommender'] = np.where(df_aux['Recommender'].isna() , df_aux['Embedding'], df_aux['Recommender'])
    
    #Renomeia colunas
    df_aux["Recommender"] = df_aux["Recommender"].replace(names_dict)
    
    df_pivot = df_aux.pivot(index='Dataset', columns=['Embedding', 'Recommender'], values=comp_metric)
    df_pivot = df_pivot.reset_index()
    
    return df_pivot

In [ ]:
def create_gray_scale(df_pivot):
    
    #Cria a escala de cinza
    GS = pd.concat([
        pd.DataFrame(MinMaxScaler(feature_range=(0.5, 0.95)).fit_transform(-df_pivot.iloc[:,1:4].values.T).round(4).T),
        pd.DataFrame(MinMaxScaler(feature_range=(0.5, 0.95)).fit_transform(-df_pivot.iloc[:,4:].values.T).round(4).T),
    ], axis=1)
    GS.index = df_pivot.index
    GS.columns = df_pivot.iloc[:,1:].columns
    
    return GS

In [ ]:
def add_gain(df_optimal, df_real):
    
    real_metrics = df_real.values[:,1:]
    optimal_metrics = df_optimal.values[:,1:]
    
    gain_percentage = (((optimal_metrics - real_metrics)/real_metrics)*100).astype(float).round(1).astype(str)
    
    df_real.iloc[:,2:4] = df_real.round(4).astype(str).iloc[:,2:4] + " (+" +  gain_percentage[:,1:3] + "\%)"
    df_real.iloc[:,5:] = df_real.round(4).astype(str).iloc[:,5:] + " (+" +  gain_percentage[:,4:] + "\%)"
    
    return df_real

In [ ]:
#comp_metric = "NDCG@10"

f = open(os.path.join(results_path, "saida_tabela_unificada.txt"), "w")

names_dict = {"itemSim": "Item-Item", "weighted": "Weighted", "ALS": "User-Item", "BPR": "User-Item"}

#Seleciona apenas as colunas terminadas em @10
metrics_names = saida.filter(regex='@10').columns

for comp_metric in metrics_names:
        
    df_final = pre_proc(saida, comp_metric)
    GS = create_gray_scale(df_final)
    
    #df_final = add_gain(df_optimal, df_real)
    
    #Coloca a escala na tabela
    df_final.iloc[:,1:] = "\\cellcolor[gray]{" + GS.astype(str) + "}" + df_final.round(4).astype(str).iloc[:,1:]
        
    tabela_latex = (df_final.style.hide(axis="index")
                    .to_latex(column_format='lccc|ccc', 
                              multicol_align='c',
                              hrules='True',
                              position_float='centering',
                              caption="Comparação de diversos métodos sob " + comp_metric))

    #Centraliza a tabela
    tabela_latex = tabela_latex.replace("\\begin{tabular}", "\\resizebox{\\textwidth} {!}{ \\begin{tabular}")
    tabela_latex = tabela_latex.replace("\\end{tabular}", "\\end{tabular} }")

    results_path = "latex_results"

    if not os.path.exists(results_path):
        os.makedirs(results_path)

    f.write("\n\n\n------------------  " + "MÉTRICA: " + comp_metric + " ------------------\n\n\n")
    f.write(tabela_latex)

f.close()

# Geração de tabelas - Comparação Item2Vec - TemporalI2V

In [ ]:
f = open(os.path.join(results_path, "saida_tabela_comparacao.txt"), "w")

comp_metric = "NDCG@10"
metrics_names = saida.filter(regex=comp_metric).columns

df_aux = saida.loc[:, ["Dataset", "Recommender", comp_metric]].copy()
df_aux = df_aux[df_aux['Recommender'] != 'ALS']
df_aux = df_aux[df_aux['Recommender'] != 'ALS_itemSim']
df_aux = df_aux[df_aux['Recommender'] != 'BPR']

print(df_aux)

df_pivot = df_aux.pivot(index='Dataset', columns='Recommender', values=comp_metric)
df_pivot = df_pivot.rename_axis(None, axis=1).reset_index()

for column in df_pivot.columns:
    if(column == 'Item2Vec_itemSim' or column == 'Dataset'):
        continue
    df_pivot[column] = round((df_pivot[column] - df_pivot['Item2Vec_itemSim'])/df_pivot['Item2Vec_itemSim'] * 100, 2)

df_pivot = df_pivot.drop(columns=['Item2Vec_itemSim'])
df_pivot.iloc[:,1:] = df_pivot.iloc[:,1:].astype(str) + '\%'

tabela_comparacao = df_pivot.style.hide(axis="index").to_latex()
tabela_comparacao = tabela_comparacao.replace("_", "-")

print(tabela_comparacao)

f.close()

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler

def data_generator(num_items_per_user=20):
    
    np.random.seed(0)

    # Define the quantities for each category
    num_zeros = int(num_items_per_user * 0.20)
    num_small = int(num_items_per_user * 0.70)
    num_big = int(num_items_per_user * 0.10)

    zeros = np.zeros(num_zeros, dtype=int)
    values_small = np.random.randint(10, 21, size=num_small)  # Random integers between 10 and 20
    values_big = np.random.randint(60, 200, size=num_big)  # Random integers between 90 and 100

    # Combine all values into a single array
    data = np.concatenate([zeros, values_small, values_big])
    
    while len(data) < num_items_per_user:
        data = np.concatenate([data, np.random.randint(10, 21, size=1)])

    # Shuffle the array
    np.random.shuffle(data)

    return data

# Function to plot interaction weights
def plot_interaction_weights(num_items_per_user, alpha, w_min, weight_proposal, time_diff_threshold, decay_threshhold):
    
    def calculate_proposal_1(s1, s2, scale):
        
        w = (1 - abs(s1 - s2))
        
        if (w + scale) > 1:
            return 1
        else:    
            return np.maximum((w+scale), 0.3)
            #return np.maximum(1 - (np.log10(1/(w+scale))), 0.3)
    
    # Define weight functions
    def calculate_proposal_2(z, alpha=alpha, w_min=w_min):
        if z < 0:
            return 1
        else:
            return 1 - np.power((z / (z + 1)), alpha)

    def calculate_proposal_3(z_scores, s1, norms, alpha=alpha, w_min=w_min):
        weight_1 = [calculate_proposal_1(s1, s2, scale) for s2 in norms]
        weight_2 = [calculate_proposal_2(z) for z in z_scores]
        return (np.array(weight_1) + np.array(weight_2))/2
        
    data = {
        'user_id': np.repeat([1], num_items_per_user),
        'item_id': list(range(1, num_items_per_user + 1)),
        'timestamp': data_generator(num_items_per_user)
    }
        
    df = pd.DataFrame(data)
          
    # Calculate cumulative time for each user
    df['cumulative_time'] = df.groupby('user_id')['timestamp'].cumsum()

    # Normalize cumulative time to [w_min, 1] using MinMaxScaler
    scaler = MinMaxScaler(feature_range=(w_min, 1))
    df['normalized_cumulative'] = df.groupby('user_id')['cumulative_time'].transform(
        lambda x: scaler.fit_transform(x.values.reshape(-1, 1)).flatten()
    )
        
    # Plot weights for each user
    for user_id in df['user_id'].unique():
        user_data = df[df['user_id'] == user_id].reset_index(drop=True)
                
        indices = {
            'Middle Item': len(user_data) // 2,
            'First Item': 0,
            'Last Item': len(user_data) - 1
        }

        fig = go.Figure()

        for label, idx in indices.items():
            
            reference_norm = user_data.loc[idx, 'normalized_cumulative']
            reference_cumsum = user_data.loc[idx, 'cumulative_time']
            
            non_noise_df = user_data[user_data['timestamp'] > time_diff_threshold]
            
            user_mean = non_noise_df['timestamp'].mean()
            user_std = non_noise_df['timestamp'].std()
            
            diff_cumsum = [abs(reference_cumsum - cumsum) for cumsum in user_data['cumulative_time']]
            
            z_scores = (diff_cumsum - user_mean) / user_std
            
            scale = scaler.scale_[0] * decay_threshhold
                        
            if weight_proposal == 1:
                weights = [calculate_proposal_1(reference_norm, norm, scale) for norm in user_data['normalized_cumulative']]
            elif weight_proposal == 2:
                weights = [calculate_proposal_2(z) for z in z_scores]
            elif weight_proposal == 3:
                weights = calculate_proposal_3(z_scores, reference_norm, user_data['normalized_cumulative'].values)
            

            # Plot the weight line
            fig.add_trace(go.Scatter(
                x=user_data['cumulative_time'],
                y=weights,
                mode='lines+markers',
                name=f'{label} (User {user_id})'
            ))

            # Add a vertical dotted line at the reference item's cumulative time
            fig.add_vline(
                x=user_data.loc[idx, 'cumulative_time'],
                line=dict(color="green", dash="dot"),
                annotation_text=label,
                annotation_position="top left"
            )

        # Update layout and display
        proposal_title = f"Proposal {weight_proposal}: "
        proposal_title += f"Mean: {round(user_mean, 2)}, STD {round(user_std, 2)}"
        fig.update_layout(
            title=f'{proposal_title} (User {user_id})',
            xaxis_title='Cumulative Time',
            yaxis_title='Weight',
            yaxis=dict(range=[0, 1.1]),
        )
        fig.show()

In [ ]:
num_items_per_user = 50
alpha = 2
w_min = 0
time_diff_threshold = 10
decay_threshhold = 200

# Fazer com que os valores decaiam apenas quando a distância entre dois itens for maior do que o treshhold
for weight_proposal in range(1, 4):
    plot_interaction_weights(num_items_per_user, alpha, w_min, weight_proposal, time_diff_threshold, decay_threshhold)

In [ ]:
df = pd.read_csv('datasets/NetflixPrize/Original/interactions.csv', sep=";", header='infer')
df.columns = ["id_user", "id_item", "rating", "datetime", "timestamp"]

In [ ]:
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
sampled_users = df['id_user'].drop_duplicates().sample(frac=0.01, random_state=42)
sampled_df = df[df['id_user'].isin(sampled_users)]

In [ ]:
sampled_df.to_csv('datasets/NetflixPrize/interactions.csv', sep=";")

In [ ]:
df = pd.read_csv('datasets/CiaoDVD/interactions_explicit.csv', sep=";", header='infer')
df.columns = ["id_user", "id_item", "rating", "datetime"]
df = df[df['rating'] > 3]
df = df.drop(columns = 'rating')

In [ ]:
df.to_csv('datasets/CiaoDVD/interactions.csv', sep=";", index=False)